# DenseNet Model
Bu notebook DenseNet modelinin eğitimi ve değerlendirmesi için oluşturulmuştur.

In [ ]:
from torchvision import models
import torch.nn as nn

densenet_model = models.densenet121(weights=None)

# Grayscale (1-channel) input için
densenet_model.features.conv0 = nn.Conv2d(
    1,
    64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

# Binary classification için output layer
densenet_model.classifier = nn.Linear(
    densenet_model.classifier.in_features,
    2
)

densenet_model = densenet_model.to(device)

print(densenet_model)

: 

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    densenet_model.parameters(),
    lr=0.0001
)

num_epochs = 2

densenet_train_losses = []
densenet_valid_losses = []

densenet_train_accuracies = []
densenet_valid_accuracies = []

for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(
        densenet_model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    valid_loss, valid_acc = validate(
        densenet_model,
        valid_loader,
        criterion,
        device
    )

    densenet_train_losses.append(train_loss)
    densenet_valid_losses.append(valid_loss)

    densenet_train_accuracies.append(train_acc)
    densenet_valid_accuracies.append(valid_acc)

    print(f"\nDenseNet Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Valid Loss: {valid_loss:.4f}")
    print(f"Valid Accuracy: {valid_acc:.2f}%")

In [ ]:
torch.save(densenet_model.state_dict(), "../models/densenet121_chest_xray.pth")
print("DenseNet121 model saved successfully.")

# DenseNet Evaluation

In [ ]:
densenet_model.eval()

densenet_all_labels = []
densenet_all_preds = []
densenet_all_probs = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = densenet_model(images)

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        densenet_all_probs.extend(probs[:, 1].cpu().numpy())
        densenet_all_preds.extend(preds.cpu().numpy())
        densenet_all_labels.extend(labels.cpu().numpy())

densenet_all_probs = np.array(densenet_all_probs)
densenet_all_preds = np.array(densenet_all_preds)
densenet_all_labels = np.array(densenet_all_labels)

print("DenseNet121 evaluation completed.")

# DenseNet Loss & Accuracy Graph

In [ ]:
plt.figure(figsize=(12,5))

# Loss
plt.subplot(1,2,1)

plt.plot(densenet_train_losses, label='Train Loss')
plt.plot(densenet_valid_losses, label='Validation Loss')

plt.title('DenseNet121 Loss Analysis')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.legend()

# Accuracy
plt.subplot(1,2,2)

plt.plot(densenet_train_accuracies, label='Train Accuracy')
plt.plot(densenet_valid_accuracies, label='Validation Accuracy')

plt.title('DenseNet121 Accuracy Analysis')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')

plt.legend()

plt.tight_layout()

plt.savefig('../outputs/densenet121_training_analysis.png')

plt.show()

# DenseNet Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(
    densenet_all_labels,
    densenet_all_preds
)

tn, fp, fn, tp = cm.ravel()

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Negative', 'Positive'],
    yticklabels=['Negative', 'Positive']
)

plt.xlabel('Predicted Label')
plt.ylabel('True Label')

plt.title('DenseNet121 Confusion Matrix')

plt.savefig('../outputs/densenet121_confusion_matrix.png')

plt.show()

print(f"TN: {tn}")
print(f"FP: {fp}")
print(f"FN: {fn}")
print(f"TP: {tp}")

# DenseNet ROC Curve

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(
    densenet_all_labels,
    densenet_all_probs
)

plt.figure(figsize=(7,6))

plt.plot(
    fpr,
    tpr,
    label=f'AUC = {densenet_auc:.4f}'
)

plt.plot([0,1], [0,1], linestyle='--')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

plt.title('DenseNet121 ROC Curve')

plt.legend()

plt.savefig('../outputs/densenet121_roc_curve.png')

plt.show()

# DenseNet Metrics

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

densenet_accuracy = accuracy_score(densenet_all_labels, densenet_all_preds)
densenet_precision = precision_score(densenet_all_labels, densenet_all_preds)
densenet_recall = recall_score(densenet_all_labels, densenet_all_preds)
densenet_f1 = f1_score(densenet_all_labels, densenet_all_preds)
densenet_auc = roc_auc_score(densenet_all_labels, densenet_all_probs)

print(f"Accuracy : {densenet_accuracy:.4f}")
print(f"Precision: {densenet_precision:.4f}")
print(f"Recall   : {densenet_recall:.4f}")
print(f"F1-Score : {densenet_f1:.4f}")
print(f"ROC-AUC  : {densenet_auc:.4f}")

### DenseNet121 Evaluation Summary

DenseNet121 was implemented as an advanced deep learning architecture for chest X-ray classification. The model was trained using CLAHE-preprocessed grayscale medical images.

Performance evaluation included accuracy, precision, recall, F1-score, confusion matrix analysis, and ROC-AUC measurement. The model demonstrated strong feature extraction capability due to its dense connectivity architecture.

DenseNet121 results were compared with ResNet18 and EfficientNet-B0 models to determine the best-performing architecture for abnormality detection in chest X-ray images.